# Content-based movie recommender — Kaggle training

This notebook trains the **same model** as the local `training/train_local.py` entry point. It needs no TMDB token and no access to your Mac.

### Inputs to add in Kaggle (right sidebar → *Add Data*)
1. Your catalogue CSV as a dataset, e.g. slug `movie-dataset` containing `all_titles.csv` (or the original `movies_2023_2026.csv` for a movies-only model), optionally a `posters/` folder.
2. The source bundle `kaggle_source.zip` as a dataset, e.g. slug `movie-recommender-source`. Create it locally with `python scripts/make_kaggle_bundle.py`.

Outputs (right sidebar → *Output*): `/kaggle/working/movie_model/` and `/kaggle/working/movie_recommender_model.zip`.

In [ ]:
import sys, zipfile
from pathlib import Path

WORKING = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
SRC = WORKING / 'src'

# Make the shared ml/ package importable inside Kaggle.
bundles = sorted(INPUT.rglob('kaggle_source.zip'))
if bundles and not (SRC / 'ml' / 'trainer.py').exists():
    SRC.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(SRC)
    print('Extracted source from', bundles[0])

for path in (SRC, WORKING / 'movie-recommender', WORKING):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print('Python paths ready')

In [ ]:
try:
    from training.train_kaggle import main as kaggle_train
except ModuleNotFoundError:
    from train_kaggle import main as kaggle_train

# Auto-detects the CSV by schema and validates it before training.
kaggle_train(['--report'])

In [ ]:
import json
from ml.artifacts import load_artifacts
from ml.inference import RecommendationEngine

artifacts = load_artifacts(WORKING / 'movie_model')
keys = ['n_movies', 'vocabulary_size', 'feature_dimensions', 'matrix_nnz',
        'training_time_seconds', 'feature_weights', 'visual_features_used']
print(json.dumps({key: artifacts.info.get(key) for key in keys}, indent=2))

engine = RecommendationEngine(artifacts)
for movie_id in [int(value) for value in list(artifacts.metadata)[:3]]:
    source = engine.get_metadata(movie_id).get('title')
    examples = [(r.title, round(r.similarity_score, 4)) for r in engine.recommend(movie_id, 5)]
    print(f'\n{source} ({movie_id}) -> {examples}')

### Download and install

1. In the Kaggle output panel, download `movie_recommender_model.zip` (also visible under `/kaggle/working/`).
2. In your local project: `python scripts/install_kaggle_model.py ~/Downloads/movie_recommender_model.zip`
3. Start the app: `python run.py` and open `http://127.0.0.1:5000/`.

No relevance labels exist in this dataset, so Precision@K / Recall@K / NDCG@K are not reported. The generated `evaluation_report.md` contains manual top-5/top-10 inspection examples and real dataset statistics.